# Magnitude vs time — rolling forecast windows

Single explanatory panel. Day 0 = start of the test set (training is negative). Colored bands = forecast windows. ★ = yearly ETAS refit. Bottom ticks = days since test start; top ticks = 1 January.


In [ ]:
%matplotlib inline
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "runnable_code" / "run_rolling_continuation.py").is_file():
    REPO_ROOT = REPO_ROOT.parent if REPO_ROOT.name == "notebooks" else next(
        parent for parent in REPO_ROOT.parents
        if (parent / "runnable_code" / "run_rolling_continuation.py").is_file()
    )

# ---- knobs ------------------------------------------------------------------
HORIZON_DAYS = 7
N_TEST_WINDOWS: int | None = 15          # None = all windows
# TRAIN_LOOKBACK_DAYS: float | None = 365 * 2
TRAIN_LOOKBACK_DAYS: float | None = 140
INVERSION_INTERVAL_DAYS: float | None = None  # None → config / 365
SAVE_DIR = Path("/a/home/cc/students/csguests/neriberman/Repos/etas/outputs/figures_for_statsei14")
SAVE_SUFFIX = None  # str appended as _{suffix} before .png, or None
# -----------------------------------------------------------------------------

def save_figure(fig, name):
    if SAVE_DIR is None:
        return
    out_dir = Path(SAVE_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = name if SAVE_SUFFIX is None else f"{name}_{SAVE_SUFFIX}"
    for ext in ("png", "svg"):
        path = out_dir / f"{stem}.{ext}"
        fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white", transparent=True)
        print(f"Wrote {path}")

ROOT = REPO_ROOT / "outputs/rolling_continuation_hauksson_pre_ridgecrest_v3"
CFG_PATH = REPO_ROOT / "config/rolling_continuation_hauksson_pre_ridgecrest_v3.json"

COLOR_TRAIN, COLOR_TEST = "#5B6B7A", "#2B2B2B"
COLOR_FINE = "#E6225B"
COLOR_BAND_A, COLOR_BAND_B = "#38BDF8", "#A78BFA"
COLOR_BOUNDARY, COLOR_DAY0 = "#0F172A", "#BE123C"
COLOR_REFIT, COLOR_YEAR = "#047857", "#475569"

cfg = json.loads(CFG_PATH.read_text(encoding="utf-8"))
mc = float(cfg["mc"])
train_start = pd.to_datetime(cfg["timewindow_start"])
test_start = pd.to_datetime(cfg["timewindow_end"])
catalog_path = REPO_ROOT / cfg["fn_catalog"]
interval_days = float(
    INVERSION_INTERVAL_DAYS
    if INVERSION_INTERVAL_DAYS is not None
    else cfg.get("inversion_interval", 365)
)

raw = pd.read_csv(catalog_path)
raw["time"] = pd.to_datetime(raw["time"], utc=True, format="mixed").dt.tz_convert(None)
raw = raw.loc[raw["magnitude"] >= mc].copy()
raw["days"] = (raw["time"] - test_start) / pd.Timedelta("1D")

plot_train_start = train_start
if TRAIN_LOOKBACK_DAYS is not None:
    plot_train_start = max(
        train_start, test_start - pd.Timedelta(days=float(TRAIN_LOOKBACK_DAYS))
    )

hdir = ROOT / f"horizon_{HORIZON_DAYS:g}d"
windows_all = pd.read_csv(hdir / "rolling_summary.csv")
windows_all["forecast_start"] = pd.to_datetime(windows_all["forecast_start"])
windows_all["forecast_end"] = pd.to_datetime(windows_all["forecast_end"])
windows_all["start_days"] = (
    windows_all["forecast_start"] - test_start
) / pd.Timedelta("1D")
windows_all["end_days"] = (
    windows_all["forecast_end"] - test_start
) / pd.Timedelta("1D")

windows = (
    windows_all if N_TEST_WINDOWS is None
    else windows_all.head(int(N_TEST_WINDOWS))
)
x_min = -float((test_start - plot_train_start) / pd.Timedelta("1D"))
x_max = float(windows["end_days"].iloc[-1])

events = raw.loc[(raw["days"] >= x_min) & (raw["days"] <= x_max)]
train = events.loc[events["days"] < 0]
test = events.loc[events["days"] >= 0]

# ETAS refit times (same rule as rolling_continuation.inversion_anchor_indices)
starts = windows_all["start_days"].to_numpy(dtype=float)
refits = [float(starts[0])]
anchor = refits[0]
for t in starts[1:]:
    if float(t) - anchor > interval_days:
        anchor = float(t)
        refits.append(anchor)
refit_days = np.asarray(
    [d for d in refits if x_min <= d <= x_max], dtype=float
)

# Year starts (1 Jan) in view
t0 = test_start + pd.Timedelta(days=float(x_min))
t1 = test_start + pd.Timedelta(days=float(x_max))
year0 = t0.year if (t0.month == 1 and t0.day == 1) else t0.year + 1
year_days, year_labels = [], []
for y in range(year0, t1.year + 1):
    d = float((pd.Timestamp(y, 1, 1) - test_start) / pd.Timedelta("1D"))
    if x_min <= d <= x_max:
        year_days.append(d)
        year_labels.append(str(y))


In [ ]:
FONT_SIZE=15


In [ ]:
fig_rolling_windows, ax_rolling_windows = plt.subplots(figsize=(20, 5.4))
fig_rolling_windows.subplots_adjust(right=0.78)
ax_rolling_windows.set_facecolor("#F8FAFC")

ax_rolling_windows.axvspan(x_min, 0.0, facecolor=COLOR_TRAIN, alpha=0.08, zorder=0)
ax_rolling_windows.axvspan(0.0, x_max, facecolor=COLOR_TEST, alpha=0.06, zorder=0)

for i, row in enumerate(windows.itertuples(index=False)):
    ax_rolling_windows.axvspan(
        float(row.start_days), float(row.end_days),
        facecolor=COLOR_BAND_A if i % 2 == 0 else COLOR_BAND_B,
        alpha=0.09, edgecolor="none", zorder=0,
    )
bounds = np.unique(np.concatenate([
    windows["start_days"].to_numpy(float),
    windows["end_days"].to_numpy(float)[-1:],
]))
for x in bounds:
    ax_rolling_windows.axvline(x, color=COLOR_BOUNDARY, lw=0.7, alpha=0.5, zorder=1)

ax_rolling_windows.axvline(0.0, color=COLOR_DAY0, lw=2.0, zorder=3)

def marker_size(mag):
    # return np.clip(18.0 * (np.asarray(mag, dtype=float) - mc + 0.4) ** 1.6, 8.0, 220.0)
    # return np.asarray(mag, dtype=float) ** 3.2
    return 3.3**(1*np.asarray(mag, dtype=float)) - mc + 0.4

ax_rolling_windows.scatter(
    train["days"], train["magnitude"],
    s=marker_size(train["magnitude"]), c=COLOR_TRAIN,
    alpha=0.55, linewidths=0, zorder=4, label="Training",
)
ax_rolling_windows.scatter(
    test["days"], test["magnitude"],
    s=marker_size(test["magnitude"]), c=COLOR_TEST,
    alpha=0.75, linewidths=0, zorder=5, label="Test",
)

ax_rolling_windows.set_xlim(x_min, x_max)
y_lo = float(events["magnitude"].min()) - 0.15
y_hi = float(events["magnitude"].max()) + 0.4
ax_rolling_windows.set_ylim(y_lo, y_hi)
ax_rolling_windows.set_xlabel("Days since beginning of test set", fontsize=FONT_SIZE)
ax_rolling_windows.set_title(
    f"T = {HORIZON_DAYS:g} d  ·  {len(windows)} windows"
)
# Open left and right so the timeline reads as continuing past the frame.
ax_rolling_windows.spines["left"].set_visible(False)
ax_rolling_windows.spines["right"].set_visible(False)
mag_ticks = np.arange(np.ceil(y_lo * 2) / 2, y_hi, 0.5)
ax_rolling_windows.set_yticks(mag_ticks)
ax_rolling_windows.tick_params(axis="y", which="both", length=0, labelleft=False, labelright=False)
ax_rolling_windows.grid(True, axis="y", alpha=0.35, color="#94A3B8")
for mag in mag_ticks:
    ax_rolling_windows.annotate(
        f"{mag:g}",
        xy=(0.01, mag),
        xycoords=("axes fraction", "data"),
        xytext=(0, 1),
        textcoords="offset points",
        ha="left", va="bottom",
        fontsize=FONT_SIZE, color="#475569",
        zorder=8, clip_on=True,
    )

# Bottom axis: day-number ticks (days since test start).
day_step = max(30.0, round((x_max - x_min) / 10.0 / 30.0) * 30.0)
day_ticks = np.arange(np.ceil(x_min / day_step) * day_step, x_max + 1e-9, day_step)
if 0.0 not in day_ticks and x_min <= 0.0 <= x_max:
    day_ticks = np.sort(np.append(day_ticks, 0.0))
ax_rolling_windows.set_xticks(day_ticks)
ax_rolling_windows.set_xticklabels([f"{t:g}" for t in day_ticks])
ax_rolling_windows.tick_params(axis="x", which="major", length=6, labelsize=FONT_SIZE)

# Top axis: first of each month. January is the year; other months are the name.
t_left = test_start + pd.Timedelta(days=float(x_min))
t_right = test_start + pd.Timedelta(days=float(x_max))
month_cursor = pd.Timestamp(year=t_left.year, month=t_left.month, day=1)
if month_cursor < t_left.normalize():
    month_cursor = month_cursor + pd.DateOffset(months=1)
month_days, month_labels = [], []
while month_cursor <= t_right:
    day = float((month_cursor - test_start) / pd.Timedelta("1D"))
    if x_min <= day <= x_max:
        month_days.append(day)
        month_labels.append(
            str(month_cursor.year) if month_cursor.month == 1 else month_cursor.strftime("%b")
        )
        if month_cursor.month == 1:
            ax_rolling_windows.axvline(day, color=COLOR_YEAR, lw=1.0, ls=":", alpha=0.55, zorder=2)
    month_cursor = month_cursor + pd.DateOffset(months=1)
if month_days:
    ax_month_ticks = ax_rolling_windows.twiny()
    ax_month_ticks.set_xlim(x_min, x_max)
    ax_month_ticks.set_xticks(month_days)
    ax_month_ticks.set_xticklabels(month_labels, fontsize=FONT_SIZE, color=COLOR_YEAR)
    for label, text in zip(ax_month_ticks.get_xticklabels(), month_labels):
        if text.isdigit():
            label.set_fontweight("bold")
    ax_month_ticks.tick_params(axis="x", colors=COLOR_YEAR, length=6, width=1.0, labelsize=FONT_SIZE)
    ax_month_ticks.spines["top"].set_color(COLOR_YEAR)
    ax_month_ticks.spines["left"].set_visible(False)
    ax_month_ticks.spines["right"].set_visible(False)

ax_rolling_windows.legend(
    handles=ax_rolling_windows.get_legend_handles_labels()[0] + [
        mpatches.Patch(facecolor=COLOR_BAND_A, alpha=0.35, label="Forecast window"),
        mpatches.Patch(facecolor=COLOR_BAND_B, alpha=0.35, label="Next window"),
    ],
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    borderaxespad=0.0,
    framealpha=0.95,
)

save_figure(fig_rolling_windows, "mag_vs_days")

plt.show()


## Realization on the same axes

Copies the figure above and adds one forecast catalog. `REALIZATION_METHOD` is `FINE` or `etas`. `REALIZATION_INDEX` is the 0-based serial number among saved seeds. `OVERLAY_ON_TEST = False` drops the test earthquakes.


In [ ]:
import pickle

from IPython.display import display

# Copy of fig_rolling_windows / ax_rolling_windows from the cell above.
fig_realization = pickle.loads(pickle.dumps(fig_rolling_windows))
ax_realization = fig_realization.axes[0]
probe = plt.figure()
canvas_cls = type(probe.canvas)
plt.close(probe)
fig_realization.set_canvas(canvas_cls(fig_realization))


REALIZATION_METHOD = "FINE"   # "FINE" or "etas"
REALIZATION_INDEX = 2
OVERLAY_ON_TEST = True

method_key = {"fine": "FINE", "etas": "etas"}[REALIZATION_METHOD.strip().lower()]
step0 = int(windows["step_index"].iloc[0])
seeds = sorted(
    int(path.name.split("_", 1)[1])
    for path in (hdir / f"step_{step0:03d}" / method_key).glob("inv_*/seed_*")
    if path.is_dir()
)
seed = seeds[int(REALIZATION_INDEX)]

frames = []
for step_index in windows["step_index"].astype(int):
    matches = sorted(
        (hdir / f"step_{step_index:03d}" / method_key).glob(
            f"inv_*/seed_{seed}/forecast_catalog.csv"
        )
    )
    if matches:
        frames.append(pd.read_csv(matches[0], usecols=["time", "magnitude"]))
realization = pd.concat(frames, ignore_index=True)
realization["time"] = pd.to_datetime(realization["time"], utc=True, format="mixed").dt.tz_convert(None)
realization["days"] = (realization["time"] - test_start) / pd.Timedelta("1D")

if not OVERLAY_ON_TEST:
    for coll in list(ax_realization.collections):
        if coll.get_label() == "Test":
            coll.remove()
added = ax_realization.scatter(
    realization["days"], realization["magnitude"],
    s=marker_size(realization["magnitude"]),
    c={"FINE": COLOR_FINE, "etas": "seagreen"}[method_key],
    alpha=0.8, linewidths=0, zorder=6,
    label=f"{method_key} realization {int(REALIZATION_INDEX)} (seed {seed})",
)
handles = [
    h for h in ax_realization.get_legend().legend_handles
    if OVERLAY_ON_TEST or h.get_label() != "Test"
]
ax_realization.legend(
    handles=handles + [added],
    loc="upper left", bbox_to_anchor=(1.02, 1.0),
    borderaxespad=0.0, framealpha=0.95,
)
ax_realization.set_title(
    f"{method_key} realization {int(REALIZATION_INDEX)} (seed {seed})"
    + (" on test events" if OVERLAY_ON_TEST else " instead of test events")
)
save_figure(fig_realization, "realization")
display(fig_realization)


## Slide backgrounds

Three pale figures from this experiment, meant to sit under a title. The empty band above the data is intentional. Run the plot cell above first; these reuse its catalog, horizon, and day-0 origin.


### Triggered versus background

Observed test earthquakes, colored by the cached soft background share \(P_0 = \mu / \lambda\). Warm orange is triggered (low \(P_0\)); cyan is background (high \(P_0\)).


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize

bg_path = (
    ROOT / f"horizon_{HORIZON_DAYS:g}d" / "analysis_cache" / "background_probability.csv"
)
p0_frame = pd.read_csv(bg_path)
p0_obs = p0_frame.loc[p0_frame["source"] == "observed"].copy()
origin_days = float((test_start - pd.Timestamp("1970-01-01")) / pd.Timedelta("1D"))
p0_obs["days"] = p0_obs["time_days"] - origin_days
p0_obs = p0_obs.sort_values("p0", ascending=False)

trig_cmap = LinearSegmentedColormap.from_list(
    "triggered_background", ["#C2410C", "#D6D3D1", "#67E8F9"]
)

fig, ax = plt.subplots(figsize=(15, 6.2), constrained_layout=True)
ax.set_facecolor("white")
sizes = np.clip(14.0 * (p0_obs["magnitude"].to_numpy() - mc + 0.35) ** 1.7, 6.0, 180.0)
sc = ax.scatter(
    p0_obs["days"], p0_obs["magnitude"],
    c=p0_obs["p0"], cmap=trig_cmap, norm=Normalize(0.0, 1.0),
    s=sizes, alpha=0.55, linewidths=0, zorder=3,
)
y_top = float(p0_obs["magnitude"].max()) + 3.2
ax.set_ylim(float(p0_obs["magnitude"].min()) - 0.2, y_top)
ax.set_xlim(float(p0_obs["days"].min()) - 5, float(p0_obs["days"].max()) + 5)
ax.set_xlabel("Days since beginning of test set")
ax.set_ylabel("Magnitude")
ax.tick_params(colors="#64748B")
for spine in ax.spines.values():
    spine.set_color("#E2E8F0")
cbar = fig.colorbar(sc, ax=ax, pad=0.01, fraction=0.03)
cbar.set_label(r"$P_0 = \mu / \lambda$")
cbar.outline.set_visible(False)
ax.set_title("Test catalog — triggered (orange) vs background (cyan)", loc="left", color="#334155")
save_figure(fig, "triggered_vs_background")
plt.show()


### One forecast, observed on top

A single ETAS seed from the analysis cache, drawn as pale dust. Observed test events sit on top, slightly stronger.


In [ ]:
DECOR_SEED = 10

cache_events = ROOT / f"horizon_{HORIZON_DAYS:g}d" / "analysis_cache" / "events"
seed_npz = np.load(cache_events / "etas" / f"seed_{DECOR_SEED}.npz")
obs_npz = np.load(cache_events / "observed.npz")
origin_days = float((test_start - pd.Timestamp("1970-01-01")) / pd.Timedelta("1D"))

sim_days = seed_npz["time_days"] - origin_days
obs_days = obs_npz["time_days"] - origin_days
sim_mag = seed_npz["magnitude"]
obs_mag = obs_npz["magnitude"]

fig, ax = plt.subplots(figsize=(15, 6.2), constrained_layout=True)
ax.set_facecolor("white")
ax.scatter(
    sim_days, sim_mag,
    s=np.clip(10.0 * (sim_mag - mc + 0.3) ** 1.5, 4.0, 90.0),
    c="#94A3B8", alpha=0.28, linewidths=0, zorder=2,
    label=f"ETAS seed {DECOR_SEED}",
)
ax.scatter(
    obs_days, obs_mag,
    s=np.clip(16.0 * (obs_mag - mc + 0.35) ** 1.6, 8.0, 160.0),
    c="#C2410C", alpha=0.72, linewidths=0, zorder=4,
    label="Observed test",
)
mag_all = np.concatenate([sim_mag, obs_mag])
day_all = np.concatenate([sim_days, obs_days])
ax.set_ylim(float(mag_all.min()) - 0.2, float(mag_all.max()) + 3.2)
ax.set_xlim(float(day_all.min()) - 5, float(day_all.max()) + 5)
ax.set_xlabel("Days since beginning of test set")
ax.set_ylabel("Magnitude")
ax.tick_params(colors="#64748B")
for spine in ax.spines.values():
    spine.set_color("#E2E8F0")
ax.legend(frameon=False, loc="upper right")
ax.set_title(
    f"ETAS seed {DECOR_SEED} (pale) and observed test events",
    loc="left", color="#334155",
)
save_figure(fig, "forecast_and_observed")
plt.show()


### Catalog map

Training and test earthquakes inside the Hauksson study polygon, sized by magnitude and washed out so a title can sit on the empty margin.


In [ ]:
cache_events = ROOT / f"horizon_{HORIZON_DAYS:g}d" / "analysis_cache" / "events"
train_npz = np.load(cache_events / "training.npz")
obs_npz = np.load(cache_events / "observed.npz")
polygon = np.load(REPO_ROOT / cfg["shape_coords"])
if not np.allclose(polygon[0], polygon[-1]):
    polygon = np.vstack([polygon, polygon[:1]])

fig, ax = plt.subplots(figsize=(12, 7.2), constrained_layout=True)
ax.set_facecolor("white")
ax.scatter(
    train_npz["longitude"], train_npz["latitude"],
    s=np.clip(4.0 * (train_npz["magnitude"] - mc + 0.4) ** 1.8, 2.0, 80.0),
    c="#94A3B8", alpha=0.22, linewidths=0, zorder=2, label="Training",
)
ax.scatter(
    obs_npz["longitude"], obs_npz["latitude"],
    s=np.clip(6.0 * (obs_npz["magnitude"] - mc + 0.4) ** 1.8, 3.0, 120.0),
    c="#C2410C", alpha=0.45, linewidths=0, zorder=3, label="Test",
)
ax.plot(polygon[:, 1], polygon[:, 0], color="#334155", lw=1.0, zorder=4)
lat_mid = float(np.mean(polygon[:, 0]))
ax.set_aspect(1.0 / np.cos(np.deg2rad(lat_mid)))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.tick_params(colors="#64748B")
for spine in ax.spines.values():
    spine.set_color("#E2E8F0")
ax.legend(frameon=False, loc="lower left")
ax.set_title("Hauksson study region", loc="left", color="#334155")
save_figure(fig, "catalog_map")
plt.show()


## Opening figures

Full 7-day test, for the first slides. ETAS is navy and FINE is rose, from the FINE slide theme. Outside the number-test band, ETAS moves toward blue and FINE toward amber so the four cases stay distinct.

The cumulative panel is the walk-forward total. Lines are expected counts, and each ribbon is the middle 80% of seeds. The magnitude panel is one row per realization. The number-test panel is Poisson δ₂ against forecast start. The band washes from rose at the lower rail to navy at the upper one, and each marker deepens as it nears 0.025 or 0.975. Marker area follows how many earthquakes fell in that window.

These cells use the full horizon cache, not the shortened window strip above. Run the first code cell first so `ROOT`, `HORIZON_DAYS`, and `SAVE_DIR` exist.


In [ ]:
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import gaussian_kde

# Opening palette. Method colors match fine_slides_figures STYLE.
# Outside-band colors are shifted just enough to separate four cases on a slide.
INK = "#1A1A1A"
ETAS = "#284473"
FINE = "#E6225B"
ETAS_OUT = "#3B82F6"
FINE_OUT = "#E8892D"
ETAS_MIST = "#D5E2F0"
FINE_MIST = "#F8D0DC"
PAPER = "#FBF9F6"
REF = "#8E887E"
MUTED = "#6A645C"
LOW_Q, HIGH_Q = 0.025, 0.975

OPENING_CACHE = ROOT / f"horizon_{HORIZON_DAYS:g}d" / "analysis_cache"
opening_scores = pd.read_csv(
    OPENING_CACHE / "scores.csv", parse_dates=["forecast_start", "forecast_end"],
)
opening_real = pd.read_csv(OPENING_CACHE / "window_realization_counts.csv")
opening_steps = (
    opening_scores.drop_duplicates("step_index").sort_values("step_index")
)
opening_ends = opening_steps.set_index("step_index")["forecast_end"]


def save_opening(fig, name):
    if SAVE_DIR is None:
        return
    out_dir = Path(SAVE_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = name if SAVE_SUFFIX is None else f"{name}_{SAVE_SUFFIX}"
    for ext in ("png", "svg"):
        path = out_dir / f"{stem}.{ext}"
        fig.savefig(
            path, dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor(), transparent=False,
        )
        print(f"Wrote {path}")


def _paper(fig, axes):
    fig.patch.set_facecolor(PAPER)
    for ax in np.atleast_1d(axes).ravel():
        ax.set_facecolor(PAPER)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(labelsize=11, length=3.5, width=0.8, colors=INK)
        ax.xaxis.label.set_color(INK)
        ax.yaxis.label.set_color(INK)


def _dates(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 5, 9]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    for label in ax.get_xticklabels():
        label.set_ha("center")


def _glow(ax, x, y, color, lw=2.6, z=4):
    ax.plot(x, y, color=color, lw=lw + 6, alpha=0.07, solid_capstyle="round", zorder=z)
    ax.plot(x, y, color=color, lw=lw + 2.2, alpha=0.18, solid_capstyle="round", zorder=z)
    ax.plot(x, y, color=color, lw=lw, solid_capstyle="round", zorder=z + 1)


def _seed_band(method):
    """Cumulative expected count, 10th to 90th percentile across seeds."""
    subset = opening_real.loc[
        opening_real["method"] == method, ["step_index", "seed", "n_forecast"]
    ]
    wide = subset.pivot_table(
        index="step_index", columns="seed", values="n_forecast", aggfunc="first",
    ).sort_index()
    cumulative = np.cumsum(wide.to_numpy(dtype=float), axis=0)
    low, high = np.nanpercentile(cumulative, [10, 90], axis=1)
    return opening_ends.loc[wide.index].to_numpy(), low, high


In [ ]:
fig, ax = plt.subplots(figsize=(13.4, 6.4))
_paper(fig, ax)

x_obs = opening_steps["forecast_end"].to_numpy()
y_obs = opening_steps["n_observed"].cumsum().to_numpy(dtype=float)
# FINE's ribbon is wider, so draw it first and let the ETAS ribbon sit on top.
for method, color, alpha in (("FINE", FINE, 0.16), ("etas", ETAS, 0.22)):
    x, low, high = _seed_band(method)
    ax.fill_between(x, low, high, color=color, alpha=alpha, linewidth=0, zorder=1)
for method, color in (("FINE", FINE), ("etas", ETAS)):
    group = opening_scores.loc[opening_scores["method"] == method].sort_values("step_index")
    _glow(ax, group["forecast_end"], group["n_forecast"].cumsum().to_numpy(dtype=float), color, lw=2.7, z=3)
_glow(ax, x_obs, y_obs, INK, lw=2.35, z=5)

_, fine_low, fine_high = _seed_band("FINE")
_, etas_low, etas_high = _seed_band("etas")
top = max(float(y_obs.max()), float(fine_high.max()), float(etas_high.max()))
ax.set_ylim(0, top * 1.04)
ax.set_xlim(pd.Timestamp(x_obs[0]), pd.Timestamp(x_obs[-1]) + pd.Timedelta(days=18))
ax.annotate(
    f"{y_obs[-1]:,.0f}",
    xy=(x_obs[-1], y_obs[-1]),
    xytext=(6, 0),
    textcoords="offset points",
    color=INK,
    fontsize=12,
    va="center",
    fontweight="medium",
)
ax.legend(
    handles=[
        Line2D([0], [0], color=INK, lw=2.3, label="Observed"),
        Line2D([0], [0], color=ETAS, lw=2.7, label="ETAS expected"),
        Patch(facecolor=to_rgba(ETAS, 0.38), edgecolor="none", label="ETAS  10–90% of seeds"),
        Line2D([0], [0], color=FINE, lw=2.7, label="FINE expected"),
        Patch(facecolor=to_rgba(FINE, 0.38), edgecolor="none", label="FINE  10–90% of seeds"),
    ],
    frameon=False, fontsize=11, loc="upper left",
)
ax.set_title(
    f"Cumulative earthquakes   ·   {HORIZON_DAYS:g}-day windows",
    loc="left", fontsize=17, pad=10, fontweight="medium", color=INK,
)
ax.set_ylabel("Cumulative event count", fontsize=13)
ax.set_xlabel("End of forecast window", fontsize=13)
_dates(ax)
fig.tight_layout()
save_opening(fig, "opening_cumulative_counts")


In [ ]:
def _load_seed_magnitudes(method):
    folder = OPENING_CACHE / "events" / method
    series = []
    for path in sorted(folder.glob("seed_*.npz"), key=lambda item: int(item.stem.split("_")[1])):
        values = np.load(path)["magnitude"].astype(float)
        values = values[np.isfinite(values)]
        if values.size:
            series.append(values)
    return series


def _mean_density(series, edges, lo, hi):
    rows = []
    for values in series:
        clipped = values[(values >= lo) & (values <= hi)]
        rows.append(np.histogram(clipped, bins=edges, density=True)[0])
    stack = np.vstack(rows)
    return stack.mean(axis=0), stack.std(axis=0, ddof=1)


obs_mag = np.load(OPENING_CACHE / "events" / "observed.npz")["magnitude"].astype(float)
etas_mags = _load_seed_magnitudes("etas")
fine_mags = _load_seed_magnitudes("FINE")

# Shared presentation window. A few ETAS extremes sit past this edge.
MAG_LO, MAG_HI = 2.35, 6.15
mag_edges = np.linspace(MAG_LO, MAG_HI, 42)
obs_density = np.histogram(
    obs_mag[(obs_mag >= MAG_LO) & (obs_mag <= MAG_HI)],
    bins=mag_edges, density=True,
)[0]
centers = 0.5 * (mag_edges[:-1] + mag_edges[1:])

etas_cmap = LinearSegmentedColormap.from_list(
    "etas_seeds", ["#9ED0F0", "#4F8FC4", "#284473", "#14263F"],
)
fine_cmap = LinearSegmentedColormap.from_list(
    "fine_seeds", ["#F8C8A0", "#F06A4A", "#E6225B", "#8E0F3A"],
)

fig, axes = plt.subplots(1, 2, figsize=(13.6, 5.9), sharey=True)
fig.subplots_adjust(wspace=0.06, top=0.82, bottom=0.12, left=0.07, right=0.98)
_paper(fig, axes)
fig.text(
    0.07, 0.96, "Magnitude by realization",
    fontsize=17, fontweight="medium", color=INK, ha="left", va="top",
)
fig.text(
    0.07, 0.905,
    "One thin histogram per seed  ·  thick curve is the mean  ·  black is observed",
    fontsize=11, color=MUTED, ha="left", va="top",
)

for ax, series, cmap, deep, name in (
    (axes[0], etas_mags, etas_cmap, ETAS, "ETAS"),
    (axes[1], fine_mags, fine_cmap, FINE, "FINE"),
):
    n_seeds = len(series)
    for index, values in enumerate(series):
        tint = index / max(n_seeds - 1, 1)
        color = cmap(0.08 + 0.92 * tint)
        clipped = values[(values >= MAG_LO) & (values <= MAG_HI)]
        counts = np.histogram(clipped, bins=mag_edges, density=True)[0]
        ax.stairs(counts, mag_edges, fill=True, color=to_rgba(color, 0.05), zorder=1)
        ax.hist(
            values, bins=mag_edges, density=True, histtype="step",
            range=(MAG_LO, MAG_HI), color=color, lw=1.05, alpha=0.55, zorder=2,
        )
    mean, spread = _mean_density(series, mag_edges, MAG_LO, MAG_HI)
    ax.fill_between(
        centers, np.maximum(mean - spread, 1e-6), mean + spread,
        step="mid", color=deep, alpha=0.14, zorder=3, linewidth=0,
    )
    ax.stairs(mean, mag_edges, color=deep, lw=2.8, zorder=4)
    ax.stairs(obs_density, mag_edges, color=INK, lw=2.2, zorder=5)
    ax.set_yscale("log")
    ax.set_xlim(MAG_LO, MAG_HI)
    ax.set_xlabel("Magnitude", fontsize=13)
    ax.set_title(name, loc="left", fontsize=14, color=deep, fontweight="medium", pad=8)
    ax.legend(
        handles=[
            Line2D([0], [0], color=INK, lw=2.2, label="Observed"),
            Line2D([0], [0], color=deep, lw=2.8, label="Mean"),
            Line2D([0], [0], color=cmap(0.55), lw=1.2, alpha=0.9, label="Seeds"),
        ],
        frameon=False, fontsize=10, loc="upper right",
    )

axes[0].set_ylabel("Density", fontsize=13)
ymin = max(1e-3, float(obs_density[obs_density > 0].min()) * 0.4)
axes[0].set_ylim(ymin, None)
save_opening(fig, "opening_magnitude_realizations")


In [ ]:
def _rgb(hex_color):
    return np.array(to_rgba(hex_color)[:3])


def _shaded(light, deep, score, alpha_near, alpha_far):
    """Deeper and more opaque as a point approaches the 0.025 or 0.975 rail."""
    depth = np.clip(np.abs(score - 0.5) / 0.475, 0.0, 1.0)
    blend = 0.55 + 0.45 * depth
    alpha = alpha_near + (alpha_far - alpha_near) * depth
    light_rgb, deep_rgb = _rgb(light), _rgb(deep)
    return [
        (*(light_rgb * (1.0 - amount) + deep_rgb * amount), float(value))
        for amount, value in zip(blend, alpha)
    ]


frame = opening_scores.dropna(subset=["poisson_delta2"]).copy()
fig, ax = plt.subplots(figsize=(13.4, 6.15))
_paper(fig, ax)

x_left = mdates.date2num(frame["forecast_start"].min() - pd.Timedelta(days=40))
x_right = mdates.date2num(frame["forecast_start"].max() + pd.Timedelta(days=18))
wash = np.linspace(0.0, 1.0, 480).reshape(-1, 1)
ax.imshow(
    wash,
    aspect="auto",
    origin="lower",
    interpolation="bilinear",
    cmap=LinearSegmentedColormap.from_list(
        "number_band", ["#F6DDE6", "#FBF8F4", "#E3EDF8"],
    ),
    extent=[x_left, x_right, LOW_Q, HIGH_Q],
    zorder=0,
    alpha=0.92,
)
ax.axhline(LOW_Q, color="#C98698", lw=0.85, ls=(0, (3.2, 3.0)), alpha=0.7, zorder=1)
ax.axhline(HIGH_Q, color="#7E97BE", lw=0.85, ls=(0, (3.2, 3.0)), alpha=0.7, zorder=1)
ax.set_xlim(x_left, x_right)

# light, deep, alpha near the middle, alpha at the rails, halo strength, halo size
layers = (
    ("etas", True, "o", False, "#5C82B4", "#1A3054", 0.72, 0.94, 3, "ETAS inside", 0.10, 1.9),
    ("etas", False, "o", False, "#7EADF0", "#245BB8", 0.88, 0.98, 4, "ETAS outside", 0.20, 2.5),
    ("FINE", True, "s", True, "#F3A9C0", "#BE1046", 0.78, 1.00, 5, "FINE inside", 0.11, 2.1),
    ("FINE", False, "s", True, "#F3BE7C", "#C26418", 0.90, 1.00, 6, "FINE outside", 0.20, 2.6),
)
handles = []
for method, inside, marker, hollow, light, deep, alpha_near, alpha_far, zorder, label, halo, glow in layers:
    subset = frame.loc[frame["method"] == method]
    score = subset["poisson_delta2"].to_numpy(dtype=float)
    time = mdates.date2num(subset["forecast_start"])
    sizes = 32 + 6.4 * np.maximum(subset["n_observed"].to_numpy(dtype=float), 0.0)
    finite = np.isfinite(score)
    passed = finite & (score >= LOW_Q) & (score <= HIGH_Q)
    chosen = passed if inside else finite & ~passed
    x, y, area = time[chosen], score[chosen], sizes[chosen]
    colors = _shaded(light, deep, y, alpha_near, alpha_far)
    ax.scatter(
        x, y, s=area * glow,
        c=[(red, green, blue, halo) for red, green, blue, _alpha in colors],
        marker=marker, linewidths=0, zorder=zorder,
    )
    if hollow:
        faces = [(red, green, blue, 0.20 + 0.28 * alpha) for red, green, blue, alpha in colors]
        edges = [(red, green, blue, 1.0) for red, green, blue, _alpha in colors]
        ax.scatter(
            x, y, s=area * 1.62, facecolors=faces, edgecolors=edges,
            marker=marker, linewidths=1.25, zorder=zorder + 0.4,
        )
        handles.append(ax.scatter(
            [], [], s=50, marker=marker, facecolors=to_rgba(deep, 0.34),
            edgecolors=deep, linewidths=1.25, label=label,
        ))
    else:
        ax.scatter(x, y, s=area, c=colors, marker=marker, linewidths=0, zorder=zorder + 0.4)
        ax.scatter(
            x, y, s=area * 0.38,
            c=[(red, green, blue, 0.95) for red, green, blue, _alpha in colors],
            marker=marker, linewidths=0, zorder=zorder + 0.6,
        )
        handles.append(ax.scatter(
            [], [], s=38, marker=marker, color=to_rgba(deep, 0.9),
            linewidths=0, label=label,
        ))

ax.set_ylim(-0.07, 1.09)
ax.set_ylabel(r"$\delta$", fontsize=16)
ax.set_xlabel("Forecast start", fontsize=13)
ax.set_title(
    f"Number test  —  T = {HORIZON_DAYS:g} d",
    loc="center", fontsize=17, pad=10, fontweight="medium", color=INK,
)
_dates(ax)
ax.legend(
    handles=handles, frameon=False, fontsize=11,
    loc="center left", bbox_to_anchor=(1.015, 0.55),
)
fig.tight_layout()
save_opening(fig, "opening_number_test")


## Number test — three visual styles

Same 7-day Poisson δ₂ scatter, same coordinates, same inside/outside split. ETAS is a square and FINE is a circle. Filled markers are inside the 0.025–0.975 band. Open markers are outside it. The figures themselves contain no text.

Run the first code cell first so `ROOT`, `HORIZON_DAYS`, and `SAVE_DIR` exist.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.path as mpath
from matplotlib.colors import to_rgba
import numpy as np
import pandas as pd

# Identical observations for every style. Nothing here rescales, jitters, or drops a finite score.
_number_scores = pd.read_csv(
    ROOT / f"horizon_{HORIZON_DAYS:g}d" / "analysis_cache" / "scores.csv",
    parse_dates=["forecast_start"],
)
_number_scores = _number_scores.loc[np.isfinite(_number_scores["poisson_delta2"])].copy()
_LOW_Q, _HIGH_Q = 0.025, 0.975
_inside = (
    (_number_scores["poisson_delta2"] >= _LOW_Q)
    & (_number_scores["poisson_delta2"] <= _HIGH_Q)
)
_GROUPS = (
    ("etas", True, "s", False),
    ("FINE", True, "o", False),
    ("etas", False, "s", True),
    ("FINE", False, "o", True),
)
_POINTS = {}
for _method, _is_inside, _marker, _hollow in _GROUPS:
    _rows = _number_scores.loc[
        (_number_scores["method"] == _method) & (_inside == _is_inside)
    ]
    _POINTS[(_method, _is_inside)] = {
        "time": _rows["forecast_start"].to_numpy(),
        "delta": _rows["poisson_delta2"].to_numpy(dtype=float).copy(),
        "size_n": np.maximum(_rows["n_observed"].to_numpy(dtype=float), 0.0).copy(),
        "marker": _marker,
        "hollow": _hollow,
    }

_X_LIM = (
    _number_scores["forecast_start"].min() - pd.Timedelta(days=14),
    _number_scores["forecast_start"].max() + pd.Timedelta(days=14),
)
_Y_LIM = (-0.06, 1.06)
_GRID_Y = (0.25, 0.5, 0.75)
_FIGURE_SIZE = (11.2, 5.35)
_ADJUST = dict(left=0.02, right=0.995, bottom=0.04, top=0.98)

# Define custom mid-century geometric marker paths
# Square split diagonally (ETAS style)
_sq_verts = [
    (-1, -1), (1, -1), (1, 1), (-1, 1), (-1, -1),
    (-1, -1), (1, 1)  # Diagonal split
]
_sq_codes = [
    mpath.Path.MOVETO, mpath.Path.LINETO, mpath.Path.LINETO, mpath.Path.LINETO, mpath.Path.CLOSEPOLY,
    mpath.Path.MOVETO, mpath.Path.LINETO
]
_mid_century_square = mpath.Path(_sq_verts, _sq_codes)

# Circle split vertically (FINE style)
_theta = np.linspace(0, np.pi, 25)
_circ_x = np.cos(_theta)
_circ_y = np.sin(_theta)
# Left semi-circle and right semi-circle combination path or distinct geometric wedges
_circle_path = mpath.Path.unit_circle()


# Mid-Century Retro Palette with a clean white background
_NUMBER_STYLES = {
    "style_mid_century_geometric": {
        "face": "#FFFFFF",
        "spine": "#2C3E50",
        "spine_lw": 0.8,
        "spines": ("left", "bottom"),
        "grid": "#E5E7EB",
        "grid_lw": 0.4,
        "grid_alpha": 0.6,
        "ref": "#9CA3AF",
        "ref_lw": 0.6,
        "ref_alpha": 0.65,
        "unity": "#6B7280",
        "unity_lw": 0.9,
        "unity_alpha": 0.8,
        # Retro palette: Navy, Sage, Terracotta, Ochre
        "etas": "#1D3557",
        "etas_accent": "#457B9D",
        "etas_edge": "#111827",
        "fine": "#E07A5F",
        "fine_accent": "#F4A261",
        "fine_edge": "#7C2D12",
        "fill_alpha": 0.88,
        "hollow_alpha": 0.98,
        "fill_lw": 0.6,
        "hollow_lw": 1.2,
        "filled_base": 20.0,
        "hollow_base": 24.0,
        "size_gain": 2.1,
    }
}


def _draw_number_style(name, style):
    fig, ax = plt.subplots(figsize=_FIGURE_SIZE)
    fig.patch.set_facecolor(style["face"])
    ax.set_facecolor(style["face"])
    
    for y in _GRID_Y:
        ax.axhline(
            y, color=style["grid"], lw=style["grid_lw"],
            alpha=style["grid_alpha"], zorder=0,
        )
    for y in (_LOW_Q, _HIGH_Q):
        ax.axhline(
            y, color=style["ref"], lw=style["ref_lw"],
            ls="--", alpha=style["ref_alpha"], zorder=1,
        )
    ax.axhline(
        1.0, color=style["unity"], lw=style["unity_lw"],
        ls="--", alpha=style["unity_alpha"], zorder=1,
    )
    
    for method, is_inside, _marker, _hollow in _GROUPS:
        spec = _POINTS[(method, is_inside)]
        base = style["hollow_base"] if spec["hollow"] else style["filled_base"]
        sizes = base + style["size_gain"] * spec["size_n"]
        n = len(spec["delta"])
        
        # Apply mid-century geometric styling to symbols
        if method == "etas":
            # Squares with geometric split motif
            marker_shape = _mid_century_square
            primary_color = style["etas"]
            edge_color = style["etas_edge"]
        else:
            # Circles with clean geometric proportion
            marker_shape = "o" if _hollow else "o"
            primary_color = style["fine"]
            edge_color = style["fine_edge"]

        if spec["hollow"]:
            artist = ax.scatter(
                spec["time"], spec["delta"], s=sizes, marker=marker_shape,
                facecolors="none",
                edgecolors=np.tile(to_rgba(edge_color, style["hollow_alpha"]), (n, 1)),
                linewidths=style["hollow_lw"], zorder=4,
            )
        else:
            artist = ax.scatter(
                spec["time"], spec["delta"], s=sizes, marker=marker_shape,
                c=np.tile(to_rgba(primary_color, style["fill_alpha"]), (n, 1)),
                edgecolors=np.tile(to_rgba(edge_color, 0.95), (n, 1)),
                linewidths=style["fill_lw"], zorder=3,
            )
        artist.set_gid(f"{method}|{int(is_inside)}")
        
    ax.set_xlim(*_X_LIM)
    ax.set_ylim(*_Y_LIM)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    
    if ax.get_legend() is not None:
        ax.get_legend().remove()
        
    for side in ("left", "right", "top", "bottom"):
        ax.spines[side].set_visible(side in style["spines"])
        ax.spines[side].set_color(style["spine"])
        ax.spines[side].set_linewidth(style["spine_lw"])
        
    fig.subplots_adjust(**_ADJUST)
    return fig, ax


def _plotted_offsets(ax):
    found = {}
    for artist in ax.collections:
        gid = artist.get_gid()
        if not gid:
            continue
        found[gid] = np.asarray(artist.get_offsets(), dtype=float).copy()
    return found


_style_axes = {}
_style_figs = {}
_style_limits = {}
for _name, _style in _NUMBER_STYLES.items():
    _fig, _ax = _draw_number_style(_name, _style)
    _style_figs[_name] = _fig
    _style_axes[_name] = _plotted_offsets(_ax)
    _style_limits[_name] = (
        tuple(float(v) for v in _ax.get_xlim()),
        tuple(float(v) for v in _ax.get_ylim()),
    )
    _texts = [text.get_text() for text in list(_ax.texts) + list(_fig.texts)]
    _tick_text = [lab.get_text() for lab in list(_ax.get_xticklabels()) + list(_ax.get_yticklabels())]
    if _ax.get_title() != "" or _ax.get_xlabel() != "" or _ax.get_ylabel() != "":
        raise RuntimeError(f"{_name} has an axis title or label")
    if _ax.get_legend() is not None or len(_ax.texts) or len(_fig.texts):
        raise RuntimeError(f"{_name} has a legend, annotation, or figure text")
    if any(piece.strip() for piece in _texts + _tick_text):
        raise RuntimeError(f"{_name} rendered text")

_reference_name = next(iter(_NUMBER_STYLES))
_reference = _style_axes[_reference_name]
_limit_reference = _style_limits[_reference_name]
_expected = {
    f"{method}|{int(is_inside)}": np.column_stack([
        mdates.date2num(_POINTS[(method, is_inside)]["time"]),
        _POINTS[(method, is_inside)]["delta"],
    ])
    for method, is_inside, _marker, _hollow in _GROUPS
}
for _name, _found in _style_axes.items():
    if _style_limits[_name] != _limit_reference:
        raise RuntimeError(f"{_name} axis limits differ from {_reference_name}")
    if set(_found) != set(_expected):
        raise RuntimeError(f"{_name} categories {set(_found)} != {set(_expected)}")
    for _key, _coords in _expected.items():
        _got = _found[_key]
        if _got.shape != _coords.shape:
            raise RuntimeError(f"{_name} {_key} shape {_got.shape} != {_coords.shape}")
        if not np.array_equal(_got, _reference[_key]):
            raise RuntimeError(f"{_name} {_key} coordinates differ from {_reference_name}")
        if not np.allclose(_got, _coords, rtol=0.0, atol=0.0):
            raise RuntimeError(f"{_name} {_key} coordinates differ from the source table")

_total = sum(len(item["delta"]) for item in _POINTS.values())
print(
    f"Verified {_total} observations, identical coordinates, "
    f"categories { {key: len(value['delta']) for key, value in _POINTS.items()} }"
)

if SAVE_DIR is not None:
    _out = Path(SAVE_DIR)
    _out.mkdir(parents=True, exist_ok=True)
    for _name, _fig in _style_figs.items():
        for _ext in ("png", "pdf", "svg"):
            _path = _out / f"{_name}.{_ext}"
            _fig.savefig(
                _path, dpi=300, facecolor=_fig.get_facecolor(),
                edgecolor="none", transparent=False,
            )
            print(f"Wrote {_path}")
